# Notebook 1. ETL and BigQuery Landing

Purpose:
- download the Argentine equity and factor data used in the project
- standardize the raw schemas
- write raw and base tables into BigQuery


## Imports and Run Settings

This notebook keeps the setup simple. We define the project settings once, then reuse the same run identifier across every table written in this run.


In [23]:
# This cell imports the main libraries and defines the run settings for the ETL notebook.
# Keeping the settings together makes reruns easier to explain and reproduce.
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import yfinance as yf

from src.bq_utils import ensure_dataset, get_bigquery_client, upload_dataframe
from src.config import (
    DATASETS,
    DEFAULT_END_DATE,
    DEFAULT_START_DATE,
    EQUITY_SYMBOLS,
    FACTOR_SYMBOLS,
    PROCESSED_TABLES,
    PROJECT_ID,
    RAW_TABLES,
)

RUN_ID = f"run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
INGESTION_TIMESTAMP = pd.Timestamp.utcnow()
START_DATE = DEFAULT_START_DATE
END_DATE = DEFAULT_END_DATE
client = get_bigquery_client()


## BigQuery Setup

Before downloading any data, we make sure the required datasets already exist in BigQuery. This keeps the notebook friendly for a first execution.


In [24]:
# This cell creates the datasets used by the project if they are still missing.
# Doing this here keeps the warehouse setup consistent across notebooks.
for dataset_name in DATASETS.values():
    ensure_dataset(client, PROJECT_ID, dataset_name)

print('Datasets ready:')
for dataset_name in DATASETS.values():
    print(f"- {PROJECT_ID}.{dataset_name}")


Datasets ready:
- bigdata-financeargentina.raw_market
- bigdata-financeargentina.processed_market
- bigdata-financeargentina.analytics_market
- bigdata-financeargentina.dbt_processed_market
- bigdata-financeargentina.serving_market


## Raw Data Download

We download the full stock universe and the market or macro proxy series that later help us interpret beta, FX sensitivity, and scenario risk.


In [25]:
# This cell downloads one equity or factor at a time and returns a flat, standardized dataframe.
# Small helper functions keep the download logic readable inside the notebook.
def _flatten_yfinance_columns(frame: pd.DataFrame) -> pd.DataFrame:
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = [col[0] if isinstance(col, tuple) else col for col in frame.columns]
    frame.columns = [str(col).strip().lower().replace(' ', '_') for col in frame.columns]
    return frame


def download_equity_history(symbol: str) -> pd.DataFrame:
    history = yf.download(
        symbol,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        progress=False,
    )
    if history.empty:
        raise ValueError(f'No price history returned for {symbol}.')

    history = _flatten_yfinance_columns(history).reset_index()
    history.columns = [str(col).strip().lower().replace(' ', '_') for col in history.columns]
    history = history.rename(columns={'adj close': 'adj_close', 'adjclose': 'adj_close'})
    history['ticker'] = symbol.upper()
    history['currency'] = 'ARS'
    history['source'] = 'yfinance'
    history['ingestion_timestamp'] = INGESTION_TIMESTAMP
    history['run_id'] = RUN_ID
    history['date'] = pd.to_datetime(history['date']).dt.date
    return history[[
        'date',
        'ticker',
        'close',
        'adj_close',
        'volume',
        'currency',
        'source',
        'ingestion_timestamp',
        'run_id',
    ]]


def download_factor_history(factor_name: str, symbol: str) -> pd.DataFrame:
    history = yf.download(
        symbol,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        progress=False,
    )
    if history.empty:
        raise ValueError(f'No factor history returned for {factor_name} ({symbol}).')

    history = _flatten_yfinance_columns(history).reset_index()
    history.columns = [str(col).strip().lower().replace(' ', '_') for col in history.columns]
    history['factor_name'] = factor_name.upper()
    history['country_risk_proxy'] = np.nan
    history['source'] = 'yfinance'
    history['ingestion_timestamp'] = INGESTION_TIMESTAMP
    history['run_id'] = RUN_ID
    history['date'] = pd.to_datetime(history['date']).dt.date
    history = history.rename(columns={'close': 'value'})
    return history[[
        'date',
        'factor_name',
        'value',
        'country_risk_proxy',
        'source',
        'ingestion_timestamp',
        'run_id',
    ]]


stock_prices = pd.concat(
    [download_equity_history(symbol) for symbol in EQUITY_SYMBOLS],
    ignore_index=True,
).sort_values(['ticker', 'date']).reset_index(drop=True)

factor_prices = pd.concat(
    [download_factor_history(name, symbol) for name, symbol in FACTOR_SYMBOLS.items()],
    ignore_index=True,
).sort_values(['factor_name', 'date']).reset_index(drop=True)


## Base Tables

The base tables stay close to the raw landing layer, but they use a consistent shape that makes Notebook 2 simpler and safer.


In [26]:
# This cell prepares the base tables used later in feature engineering.
# The idea is to keep only the columns that Notebook 2 really needs.
base_stock_prices = stock_prices.copy()
base_factor_prices = factor_prices.copy()

print('Raw stock rows:', len(stock_prices))
print('Raw factor rows:', len(factor_prices))
print('Base stock rows:', len(base_stock_prices))
print('Base factor rows:', len(base_factor_prices))

display(base_stock_prices.head())
display(base_factor_prices.head())


Raw stock rows: 17810
Raw factor rows: 9198
Base stock rows: 17810
Base factor rows: 9198


,date,ticker,close,adj_close,volume,currency,source,ingestion_timestamp,run_id
0,2019-01-02,BMA.BA,166.250000,123.107140,87918,ARS,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229
1,2019-01-03,BMA.BA,168.000000,124.403008,135188,ARS,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229
2,2019-01-04,BMA.BA,174.199997,128.994034,158136,ARS,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229
3,2019-01-07,BMA.BA,187.500000,138.842621,369658,ARS,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229
4,2019-01-08,BMA.BA,184.000000,136.250870,182837,ARS,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229


,date,factor_name,value,country_risk_proxy,source,ingestion_timestamp,run_id
0,2019-01-02,EEM,39.160000,NaN,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229
1,2019-01-03,EEM,38.450001,NaN,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229
2,2019-01-04,EEM,39.689999,NaN,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229
3,2019-01-07,EEM,39.779999,NaN,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229
4,2019-01-08,EEM,39.930000,NaN,yfinance,2026-04-26 02:12:29.014381+00:00,run_20260426_021229


## Upload to BigQuery

We now write both the raw landing tables and the base processed tables. This gives the next notebook a clean warehouse starting point.


In [27]:
# This cell writes the raw and base layers to BigQuery with full-refresh logic.
# Truncate mode keeps the academic workflow deterministic and easy to review.
upload_dataframe(
    client,
    stock_prices,
    PROJECT_ID,
    DATASETS['raw'],
    RAW_TABLES['stock_prices'],
    'WRITE_TRUNCATE',
)
upload_dataframe(
    client,
    factor_prices,
    PROJECT_ID,
    DATASETS['raw'],
    RAW_TABLES['factor_prices'],
    'WRITE_TRUNCATE',
)
upload_dataframe(
    client,
    base_stock_prices,
    PROJECT_ID,
    DATASETS['processed'],
    PROCESSED_TABLES['base_stock_prices'],
    'WRITE_TRUNCATE',
)
upload_dataframe(
    client,
    base_factor_prices,
    PROJECT_ID,
    DATASETS['processed'],
    PROCESSED_TABLES['base_factor_prices'],
    'WRITE_TRUNCATE',
)

print('Notebook 1 tables written:')
for dataset_name, table_name in [
    (DATASETS['raw'], RAW_TABLES['stock_prices']),
    (DATASETS['raw'], RAW_TABLES['factor_prices']),
    (DATASETS['processed'], PROCESSED_TABLES['base_stock_prices']),
    (DATASETS['processed'], PROCESSED_TABLES['base_factor_prices']),
]:
    print(f"- {PROJECT_ID}.{dataset_name}.{table_name}")


Notebook 1 tables written:
- bigdata-financeargentina.raw_market.stock_prices
- bigdata-financeargentina.raw_market.factor_prices
- bigdata-financeargentina.processed_market.base_stock_prices
- bigdata-financeargentina.processed_market.base_factor_prices
